# Huấn Luyện Mô Hình Reasoning LLM From Scratch trên Google Colab (GPU T4)

Notebook này hướng dẫn quy trình chạy toàn bộ từ đầu:
1. Kiểm tra GPU NVIDIA T4 (16GB VRAM)
2. Mount Google Drive để lưu Checkpoints tự động (tránh mất mát khi Colab ngắt phiên)
3. Cài đặt môi trường cần thiết
4. Huấn luyện Tokenizer Byte-Level BPE (16,384 vocab, 7 special tokens)
5. Stream & Đóng gói dữ liệu tiền huấn luyện ra file nhị phân `train.bin` (`uint16 memmap`)
6. Đo Throughput thực tế của T4
7. Khởi chạy Pre-training với FP16 AMP & Cosine LR

### 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

### 2. Mount Google Drive
Đảm bảo các checkpoint mô hình (`.pt`) được lưu an toàn trực tiếp vào Google Drive của bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/llm_checkpoints

### 3. Cài đặt Thư viện Phụ trợ

In [ ]:
!pip install -q pyyaml datasets tqdm pytest

### 4. Tải Mã Nguồn Dự Án (Nếu chưa có)

In [ ]:
# Nếu bạn đẩy code lên GitHub, clone về như sau:
# !git clone https://github.com/<your-username>/LLM-self-built.git
# %cd LLM-self-built

# Kiểm tra các unit test trước khi train
!python -m pytest -q

### 5. Huấn Luyện Tokenizer Byte-Level BPE (from scratch)
Học bảng ghép từ vựng (vocab size 16,384) và lưu vào `artifacts/tokenizer/`.

In [ ]:
!python scripts/train_tokenizer.py \
    --vocab-size 16384 \
    --tinystories-samples 50000 \
    --gsm8k-samples 7473 \
    --output-dir artifacts/tokenizer

### 6. Stream & Đóng Gói Dữ Liệu Tiền Huấn Luyện thành `train.bin`
Stream 300,000 mẫu (~250M tokens) trực tiếp từ Hugging Face qua mạng siêu tốc của Colab, tokenize và lưu thành file nhị phân `train.bin` kiểu `uint16` (~500 MB).

In [ ]:
!python scripts/pack_data.py \
    --tokenizer-dir artifacts/tokenizer \
    --hf-dataset HuggingFaceFW/fineweb-2 \
    --hf-subset vie_Latn \
    --max-samples 300000 \
    --output-bin /content/train.bin

### 7. Đo Throughput Thực Tế trên GPU T4 & Xác Định Token Budget

In [ ]:
!python scripts/benchmark_throughput.py \
    --config configs/compact_12gb.yaml \
    --micro-batch-size 4 \
    --seq-len 1024 \
    --gpu-hours-budget 8.0

### 8. Khởi Chạy Pre-training trên GPU T4 (Lưu Checkpoints vào Drive)
Mô hình cấu hình Compact (~43.8M tham số, context length 2048, FP16 AMP, Gradient Checkpointing).

In [ ]:
!python scripts/train_core.py \
    --config configs/compact_12gb.yaml \
    --train-bin /content/train.bin \
    --out-dir /content/drive/MyDrive/llm_checkpoints \
    --batch-size 4 \
    --grad-accum 16 \
    --max-steps 5000 \
    --lr 6e-4 \
    --save-interval 500

### 9. Thử Nghiệm Sinh Văn Bản Từ Checkpoint Mới Huấn Luyện

In [ ]:
import torch
from src.model.config import ModelConfig
from src.model.transformer import DecoderOnlyLM
from src.tokenizer.bpe import ByteLevelBPETokenizer
from src.inference.generator import AutoregressiveGenerator

# 1. Load config & tokenizer
cfg = ModelConfig.from_yaml("configs/compact_12gb.yaml")
tok = ByteLevelBPETokenizer.load("artifacts/tokenizer")

# 2. Load trained weights
device = "cuda" if torch.cuda.is_available() else "cpu"
model = DecoderOnlyLM(cfg).to(device).eval()
ckpt = torch.load("/content/drive/MyDrive/llm_checkpoints/step_005000.pt", map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])

# 3. Run Generator
gen = AutoregressiveGenerator(model, pad_token_id=tok.token_to_id("<|pad|>"), eos_token_id=tok.token_to_id("<|eos|>"))
prompt = "Trí tuệ nhân tạo là"
prompt_ids = torch.tensor([tok.encode(prompt, add_bos=True)], device=device)

res = gen.generate(prompt_ids, max_new_tokens=64, temperature=0.7, top_p=0.9)
output_text = tok.decode(res.sequences[0].tolist(), skip_special_tokens=True)
print("Output:\n", output_text)
print(f"\nTokens/sec: {res.tokens_per_sec:.1f} | TTFT: {res.ttft_ms:.1f} ms")